[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [14]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
      super().__init__()
      self.d_model = d_model
      self.d_ff = d_ff
      self.num_experts = num_experts
      self.top_k = top_k

      self.router = nn.Linear(d_model, num_experts)
      self.experts = nn.ModuleList(
          nn.Sequential(
              nn.Linear(d_model, d_ff),
              nn.ReLU(),
              nn.Linear(d_ff, d_model)
          ) for _ in range(num_experts)
      )

    def forward(self, x):
      B, S, d_model = x.shape
      num_tokens = B * S
      tokens = x.view(num_tokens, d_model)

      tokens_experts_scores = self.router(tokens)
      token_experts_scores, token_experts_ids = torch.topk(tokens_experts_scores, self.top_k, dim=-1)
      token_experts_scores_softmax = torch.softmax(token_experts_scores, dim=-1)

      token_experts_scores_softmax = token_experts_scores_softmax.view(-1)
      token_experts_ids = token_experts_ids.view(-1)


      token_ids = torch.repeat_interleave(torch.arange(0, num_tokens, device=x.device), repeats=self.top_k)
      # token_ids = torch.arange(0, num_tokens, device=x.device).unsqueeze(-1).expand(-1, self.top_k).reshape(-1)

      value = torch.zeros_like(tokens, device=x.device)

      for expert_idx in range(self.num_experts):
        mask = (
            token_experts_ids == expert_idx
        )

        if not mask.any():
          continue

        selected_tokens_ids = token_ids[mask]

        expert_out_for_tokens = self.experts[expert_idx](tokens[selected_tokens_ids])
        weighted_expert_out_for_tokens = token_experts_scores_softmax[mask].unsqueeze(-1) * expert_out_for_tokens

        value.scatter_add_(dim=0, index=selected_tokens_ids.unsqueeze(-1).expand(-1, d_model), src=weighted_expert_out_for_tokens)

      return value.view(B, S, d_model)


In [15]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (6.7ms)
  ✅ [2/4] Has router and experts (4.5ms)
  ✅ [3/4] Router logits shape (3.6ms)
  ✅ [4/4] Gradient flow (5.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (19.9ms total)
  Progress saved. Run status() to see your dashboard.

